# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [499]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [500]:
df = pd.read_csv(RAW)

## 1. Dados faltantes e Pré-tratamento de colunas

*Identificando colunas com valores nulos*

In [501]:
df.isnull().sum()

ID                          0
MONTHS_BALANCE              0
STATUS                      0
CODE_GENDER                 0
FLAG_OWN_CAR                0
FLAG_OWN_REALTY             0
CNT_CHILDREN                0
AMT_INCOME_TOTAL            0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
DAYS_EMPLOYED               0
FLAG_MOBIL                  0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
OCCUPATION_TYPE        240048
CNT_FAM_MEMBERS             0
dtype: int64

**Interpretação**

Há 240048 nulos na coluna OCCUPATION_TYPE

---

*Verificando valores da coluna OCCUPATION_TYPE*

In [502]:
df['OCCUPATION_TYPE'].value_counts()

OCCUPATION_TYPE
Laborers                 131572
Core staff                77112
Sales staff               70362
Managers                  67738
Drivers                   47678
High skill tech staff     31768
Accountants               27223
Medicine staff            26691
Cooking staff             13416
Security staff            12400
Cleaning staff            11399
Private service staff      6714
Low-skill Laborers         3623
Secretaries                3149
Waiters/barmen staff       2557
HR staff                   1686
IT staff                   1319
Realty agents              1260
Name: count, dtype: int64

**Interpretação**

Nota-se que os valores não nulos se referem à cargos dos candidatos contratados.

Hipótese: valores nulos são de candidatos não contratados?
________________________________________________________________________________________________________________________________

*Tratando os nulos da coluna OCCUPATION_TYPE*

**Decisão:** 

O dicionário de dados informa que, se o número na coluna DAYS_EMPLOYED for positivo, significa que a pessoa está desempregada.

Para essa condição, os nulos da coluna OCCUPATION_TYPE serão substituídos por "Unemployed".

O restante dos nulos serão substituir por "Unknown", pois não se tem informações sobre a ocupação dessas pessoas.

In [503]:
for item in df[df.DAYS_EMPLOYED > 0].index:
  df.loc[item, "OCCUPATION_TYPE"] = "Unemployed"

df.fillna('Unknown', inplace=True)

df.isnull().sum()

ID                     0
MONTHS_BALANCE         0
STATUS                 0
CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
DAYS_BIRTH             0
DAYS_EMPLOYED          0
FLAG_MOBIL             0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
OCCUPATION_TYPE        0
CNT_FAM_MEMBERS        0
dtype: int64

Não há mais valores nulos

---

*Excluindo coluna FLAG_MOBIL que contém apenas 1 valor*

In [504]:
df.drop(columns='FLAG_MOBIL', inplace=True)
df.shape

(777715, 19)

---

*Tratando o tipo de dado da coluna CNT_FAM_MEMBERS*

Alterando para int64

In [505]:
df.CNT_FAM_MEMBERS = df.CNT_FAM_MEMBERS.astype('int64')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 777715 entries, 0 to 777714
Data columns (total 19 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   777715 non-null  int64  
 1   MONTHS_BALANCE       777715 non-null  int64  
 2   STATUS               777715 non-null  str    
 3   CODE_GENDER          777715 non-null  str    
 4   FLAG_OWN_CAR         777715 non-null  str    
 5   FLAG_OWN_REALTY      777715 non-null  str    
 6   CNT_CHILDREN         777715 non-null  int64  
 7   AMT_INCOME_TOTAL     777715 non-null  float64
 8   NAME_INCOME_TYPE     777715 non-null  str    
 9   NAME_EDUCATION_TYPE  777715 non-null  str    
 10  NAME_FAMILY_STATUS   777715 non-null  str    
 11  NAME_HOUSING_TYPE    777715 non-null  str    
 12  DAYS_BIRTH           777715 non-null  int64  
 13  DAYS_EMPLOYED        777715 non-null  int64  
 14  FLAG_WORK_PHONE      777715 non-null  int64  
 15  FLAG_PHONE           777715 

## 2. Definição da variável alvo

*Tratando a coluna STATUS*

A coluna alvo será a STATUS, porém seus dados precisam ser tratados para se saber se o cliente é um bom ou mau pagador.

Adotando um critério padrão de risco:

0 - Bom Pagador: clientes que apresentam os indicadores 0, 1, C ou X.

1 - Mau Pagador: clientes que apresentam os indicadores 2, 3, 4 ou 5 em seu STATUS, demonstrando inadimplência grave, com atrasos maiores de 60 dias.



In [506]:
# Agrupando por ID
cliente = df.groupby('ID')['STATUS'].sum().reset_index()

# Separando o status que identifica mal pagador (>=60 dias de atraso)
bad_status = {'2', '3', '4', '5'}

# Substituindo os valores por 0 ou 1
cliente.STATUS = cliente.STATUS.apply(lambda s: 1 if any(char in bad_status for char in str(s)) else 0)

# Excluindo a coluna STATUS do df
df.drop(columns='STATUS', inplace=True)

# Juntando os 2 dataframes
df = pd.merge(df, cliente, on='ID', how='inner' )

df.head()

,ID,MONTHS_BALANCE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS
0,5008804,0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
1,5008804,-1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
2,5008804,-2,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
3,5008804,-3,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0
4,5008804,-4,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,Unknown,2,0


## 3. Normalização / padronização

*Tratando a coluna DAYS_EMPLOYED*

Antes de aplicar o escalonador, há necessidade de tratar os valores da coluna DAYS_EMPLOYED.

Pelo histograma da etapa 1.2, verifica-se que o range dos valores é bem grande a partir do zero.

*Verificando os valores únicos positivos*

In [507]:
df.DAYS_EMPLOYED[df.DAYS_EMPLOYED > 0].unique()

array([365243])

**Interpretação**

Nota-se que há apenas 1 valor positivo (365243) para vários clientes (indicando que o cliente está desempregado), o que se trata de uma flag.


*Alterando os valores positivos da coluna DAYS_EMPLOYED para 1, para não afetar os testes*

In [508]:
df.DAYS_EMPLOYED = df.DAYS_EMPLOYED.apply(lambda x: 1 if x > 0 else x)
df.DAYS_EMPLOYED[df.DAYS_EMPLOYED > 0].unique()

array([1])

---

**Escolha do Escalonador**


Para modelos baseados em árvores não há necessidade de escalonador, porém, para modelos baseados em distância é recomendado sua utilização.

Foi escolhido o **StandardScaler** para as colunas **numéricas**, pois essas colunas apresentam valores bem diferentes.

Aplicando o escalonador elas ficam todas com a mesma escala (com média para 0 e desvio padrão para 1).

---

*Aplicando o **StandardScaler** nas variáveis numéricas de interesse*

In [509]:
# Separando as colunas numéricas
numericas = ['AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS']

# Instanciando o escalonador
scaler = StandardScaler()

# Escalonando
df_scaled = scaler.fit_transform(df[numericas])

# Transformando em dataframe
df_scaled = pd.DataFrame(df_scaled, columns=numericas)

df_scaled.shape

(777715, 5)

*Excluindo as colunas numéricas do df original*

In [510]:
df.drop(columns=df[numericas], inplace = True)
df.shape

(777715, 14)

*Juntando os dataframes*

In [511]:
df = pd.concat([df, df_scaled], axis = 1)
df.shape

(777715, 19)

In [512]:
df.head()

,ID,MONTHS_BALANCE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,STATUS,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS
0,5008804,0,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155
1,5008804,-1,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155
2,5008804,-2,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155
3,5008804,-3,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155
4,5008804,-4,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,1,0,0,Unknown,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155


## 4. Feature engineering

Há colunas que podem ser binarizadas e há colunas categóricas podem ser tratadas para ficarem com o mesmo padrão das outras colunas.

_____

*Binarizando as colunas FLAG_OWN_CAR, FLAG_OWN_REALTY*

Conforme visto na anáise exploratória de dados, as colunas FLAG_OWN_CAR, FLAG_OWN_REALTY possuem valores dicotômicos (N e Y) que podem ser binarizados (0 e 1)

Essa binarização pode ser feita pelo Label Encoder, porém, neste caso será feito de forma manual.

In [513]:
df.FLAG_OWN_CAR = df.FLAG_OWN_CAR.apply(lambda x: 0 if x=='N' else 1)
df.FLAG_OWN_REALTY = df.FLAG_OWN_REALTY.apply(lambda x: 0 if x=='N' else 1)

----

*Aplicando o **One-Hot Encoder** nas colunas categóricas*

In [514]:
# Separando as colunas categóricas
categoricas = []

for i in df.columns:
  if df[i].dtype == 'str':
    categoricas.append(i)

print(categoricas)

# Aplicando o hot encoder
hot = []

for i in df.columns:
  hot = pd.get_dummies(df[categoricas], prefix = 'hot')


# Mesclando os  dataframes
df = pd.concat([df, hot], axis=1)

# Excluindo as colunas categóricas originais
df.drop(columns=categoricas, inplace=True)

df.head()

['CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


,ID,MONTHS_BALANCE,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,STATUS,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,CNT_CHILDREN,CNT_FAM_MEMBERS,hot_F,hot_M,hot_Commercial associate,hot_Pensioner,hot_State servant,hot_Student,hot_Working,hot_Academic degree,hot_Higher education,hot_Incomplete higher,hot_Lower secondary,hot_Secondary / secondary special,hot_Civil marriage,hot_Married,hot_Separated,hot_Single / not married,hot_Widow,hot_Co-op apartment,hot_House / apartment,hot_Municipal apartment,hot_Office apartment,hot_Rented apartment,hot_With parents,hot_Accountants,hot_Cleaning staff,hot_Cooking staff,hot_Core staff,hot_Drivers,hot_HR staff,hot_High skill tech staff,hot_IT staff,hot_Laborers,hot_Low-skill Laborers,hot_Managers,hot_Medicine staff,hot_Private service staff,hot_Realty agents,hot_Sales staff,hot_Secretaries,hot_Security staff,hot_Unemployed,hot_Unknown,hot_Waiters/barmen staff
0,5008804,0,1,1,1,0,0,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,5008804,-1,1,1,1,0,0,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,5008804,-2,1,1,1,0,0,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,5008804,-3,1,1,1,0,0,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
4,5008804,-4,1,1,1,0,0,0,2.351502,1.00381,-0.908918,-0.574026,-0.230155,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False


In [515]:
df.shape

(777715, 56)

*Excluindo coluna ID*
A coluna ID possui valores muito diversos e pode atrapalhar em alguns modelos

In [516]:
df.drop(columns='ID', inplace=True)

*Removendo linhas duplicadas*

In [517]:
df.drop_duplicates(inplace=True)

In [518]:
df.duplicated().sum()

np.int64(0)

In [520]:
df.shape

(333203, 55)

## 5. Salvar dataset tratado

In [519]:
dataset_tratado = df.copy

PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)